<a href="https://colab.research.google.com/github/dohee-jin/data-mining-final-project/blob/main/kbo_goldenglove_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 골든글러브 후보 예측

- 선수 개인 기록을 활용해 2026년 골든글러브 후보 가능성이 높은 선수를 예측한다.
- 타자와 투수는 기록 컬럼이 다르므로 분리해서 분석한다.

---

## 2. 골든글로브 후보 예측 데이터 사용 전략

데이터는 크게 세 종류로 나눈다.

| 구분 | 출처 | 직접 정리 여부 | 사용 목적 |
|---|---|---:|---|
| 과거 선수 기록 | Kaggle KBO Player Dataset 1982-2025 | X | 골든글러브 예측 학습 데이터 |
| 2026 현재 선수 기록 | KBO 공식 홈페이지 기록실 | O | 2026 골든글러브 후보 예측 대상 |
| 골든글러브 수상자 목록 | KBO 공식 골든글러브 수상 현황 | O | `is_goldenglove` 라벨 생성 |

---

## 3. 골든글러브 후보 예측용 데이터

### 3.1 Kaggle에서 가져올 데이터

Kaggle의 `KBO Player Dataset (1982-2025)`를 과거 학습 데이터로 사용한다.

사용할 데이터는 다음과 같다.

```text
kbo_batting_stats_by_season_1982-2025.csv
kbo_pitching_stats_by_season_1982-2025.csv
```

Kaggle 데이터는 1982년부터 2025년까지의 KBO 정규시즌 선수 타격 및 투수 기록을 포함한다. 실제 분석에서는 너무 오래된 데이터까지 모두 쓰기보다, 최근 야구 환경과 비슷한 기간만 필터링하여 사용한다.

추천 사용 기간:

```text
2015~2025
```

### 3.2 직접 정리해야 하는 2026 선수 기록

2026년은 아직 시즌이 진행 중이므로 Kaggle 과거 데이터에 포함되어 있지 않다. 따라서 KBO 공식 홈페이지에서 현재 기록을 복사해 CSV로 직접 정리하고 깃허브에 csv로 raw 파일을 업로드했다.

```text
data/raw/kbo_hitter_data_2026.csv
data/raw/kbo_pitcher_data_2026.csv
```

2026 선수 기록 파일에는 반드시 `year` 컬럼을 추가한다.   
타자 기록에는 골든글러브 포지션별 예측을 위해 `position` 칼럼을 추가한다.   
일부 선수는 시즌 중 여러 포지션에 출전하므로, 본 분석에서는 데이터 단순화와 모델 적용의 일관성을 위해 선수별 대표 포지션을 하나만 부여하였다. 대표 포지션은 KBO 기록실의 선수 구분 또는 주요 출전 포지션을 기준으로 정리하였다.

```text
year = 2026
position = C, 1B, 2B, 3B, SS, OF, DH
```

예시:

| year | player | team | avg | hr | rbi | ops |
|---:|---|---|---:|---:|---:|---:|
| 2026 | 선수명 | LG | 0.315 | 12 | 45 | 0.890 |

---

### 4. kaggle 데이터셋 접근 불가
kaggle 데이터셋이 내려가 접근하지 못해 해당 데이터셋으로의 모델 생성, 예측은 불가능하게 됨.  
`https://huggingface.co/datasets/juhonov/KBOresearch?utm_source=chatgpt.com` 의 선수 기록 수집기를 통해 2000-2025년도 선수 기록을 바탕으로 데이터를 수집, 학습하기로 계획을 변경함

### kagglehub로 kbo 1982-2025 데이터셋 불러오기
kaggle내 데이터셋이 삭제되어 사용불가

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("netsong/kbo-player-dataset-by-regular-season-1982-2025")

print("Path to dataset files:", path)

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

In [ ]:
import os
import pandas as pd

path = kagglehub.dataset_download("netsong/kbo-player-dataset-by-regular-season-1982-2025")

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

## 다른 데이터셋 사용

In [ ]:
!pip install -q datasets

from datasets import load_dataset
from huggingface_hub import hf_hub_download
import pandas as pd

# Hugging Face에서 JSON 파일 직접 다운로드
hitter_file = hf_hub_download(
    repo_id="juhonov/KBOresearch",
    filename="kbo_hitter_stats_2000_2025.json",
    repo_type="dataset"
)

pitcher_file = hf_hub_download(
    repo_id="juhonov/KBOresearch",
    filename="kbo_pitcher_stats_2000_2025.json",
    repo_type="dataset"
)

print(hitter_file)
print(pitcher_file)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


kbo_hitter_stats_2000_2025.json:   0%|          | 0.00/3.44M [00:00<?, ?B/s]

kbo_pitcher_stats_2000_2025.json:   0%|          | 0.00/2.74M [00:00<?, ?B/s]

/root/.cache/huggingface/hub/datasets--juhonov--KBOresearch/snapshots/14fd8e53426a5e3bd21b576fa109ea8928463008/kbo_hitter_stats_2000_2025.json
/root/.cache/huggingface/hub/datasets--juhonov--KBOresearch/snapshots/14fd8e53426a5e3bd21b576fa109ea8928463008/kbo_pitcher_stats_2000_2025.json


In [ ]:
# Json 구조 확인
import json

with open(hitter_file, "r", encoding="utf-8") as f:
    hitter_json = json.load(f)

with open(pitcher_file, "r", encoding="utf-8") as f:
    pitcher_json = json.load(f)

print(type(hitter_json))
print(type(pitcher_json))

if isinstance(hitter_json, dict):
    print(hitter_json.keys())

if isinstance(pitcher_json, dict):
    print(pitcher_json.keys())

<class 'dict'>
<class 'dict'>
dict_keys(['metadata', 'data'])
dict_keys(['metadata', 'data'])


In [ ]:
# json 데이터 구조 확인
def inspect_json(obj, name="root"):
    print(f"\n{name}: type={type(obj)}")

    if isinstance(obj, dict):
        print("keys:", list(obj.keys()))
        for k, v in obj.items():
            print(f"  {k}: {type(v)}", end="")
            if isinstance(v, list):
                print(f", list length={len(v)}")
                if len(v) > 0:
                    print("    first item type:", type(v[0]))
                    print("    first item:", v[0])
            elif isinstance(v, dict):
                print(f", dict keys={list(v.keys())[:10]}")
            else:
                print(f", value={v}")

    elif isinstance(obj, list):
        print("list length:", len(obj))
        if len(obj) > 0:
            print("first item type:", type(obj[0]))
            print("first item:", obj[0])

inspect_json(hitter_json, "hitter_json")
inspect_json(pitcher_json, "pitcher_json")


hitter_json: type=<class 'dict'>
keys: ['metadata', 'data']
  metadata: <class 'dict'>, dict keys=['description', 'source', 'collected_at', 'total_records']
  data: <class 'list'>, list length=8188
    first item type: <class 'dict'>
    first item: {'순위': '1', 'playerId': '62558', '선수명': '김준태', '팀명': 'LG', 'AVG': '1.000', 'G': '2', 'PA': '2', 'AB': '1', 'R': '1', 'H': '1', '2B': '0', '3B': '0', 'HR': '0', 'TB': '1', 'RBI': '0', 'SAC': '0', 'SF': '0', 'year': 2025, 'teamCode': 'LG', 'teamName': 'LG'}

pitcher_json: type=<class 'dict'>
keys: ['metadata', 'data']
  metadata: <class 'dict'>, dict keys=['description', 'source', 'collected_at', 'total_records']
  data: <class 'list'>, list length=5696
    first item type: <class 'dict'>
    first item: {'순위': '1', 'playerId': '50904', '선수명': '이종준', '팀명': 'LG', 'ERA': '0.00', 'G': '2', 'W': '0', 'L': '0', 'SV': '0', 'HLD': '0', 'WPCT': '-', 'IP': '1 2/3', 'H': '0', 'HR': '0', 'BB': '1', 'HBP': '0', 'SO': '2', 'R': '0', 'ER': '0', 'WHIP': '0

In [ ]:
from google.colab import files

# 데이터프레임 생성
hitter = pd.DataFrame(hitter_json["data"])
pitcher = pd.DataFrame(pitcher_json["data"])

print(hitter.shape)
print(pitcher.shape)

display(hitter.head())
display(pitcher.head())

# 데이터프레임 로컬 저장
# 로컬 저장 후 깃허브에 직접 업로드 필요
"""
hitter.to_csv("kbo_hitter_stats_2000-2025.csv", index=False, encoding="utf-8-sig")
pitcher.to_csv("kbo_pitcher_stats_2000-2025.csv", index=False, encoding="utf-8-sig")

files.download("kbo_hitter_stats_2000-2025.csv")
files.download("kbo_pitcher_stats_2000-2025.csv")
"""

(8188, 27)
(5696, 26)


,순위,playerId,선수명,팀명,AVG,G,PA,AB,R,H,...,year,teamCode,teamName,SB,CS,BB,HBP,SO,GDP,E
0,1,62558,김준태,LG,1.000,2,2,1,1,1,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,54166,김현종,LG,0.400,10,6,5,3,2,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,65207,신민재,LG,0.313,135,538,463,87,145,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,53123,오스틴,LG,0.313,116,499,425,82,133,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,68119,문성주,LG,0.305,135,542,475,57,145,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,순위,playerId,선수명,팀명,ERA,G,W,L,SV,HLD,...,SO,R,ER,WHIP,year,teamCode,teamName,CG,SHO,TBF
0,1,50904,이종준,LG,0.00,2,0,0,0,0,...,2,0,0,0.60,2025,LG,LG,NaN,NaN,NaN
1,2,77263,김강률,LG,1.46,12,1,0,1,4,...,9,2,2,1.22,2025,LG,LG,NaN,NaN,NaN
2,3,61145,이우찬,LG,1.89,23,0,1,0,0,...,20,6,4,1.42,2025,LG,LG,NaN,NaN,NaN
3,4,55167,김영우,LG,2.40,66,3,2,1,7,...,56,17,16,1.32,2025,LG,LG,NaN,NaN,NaN
4,5,50106,유영찬,LG,2.63,39,2,2,21,1,...,52,14,12,1.32,2025,LG,LG,NaN,NaN,NaN


'\nhitter.to_csv("kbo_hitter_stats_2000-2025.csv", index=False, encoding="utf-8-sig")\npitcher.to_csv("kbo_pitcher_stats_2000-2025.csv", index=False, encoding="utf-8-sig")\n\nfiles.download("kbo_hitter_stats_2000-2025.csv")\nfiles.download("kbo_pitcher_stats_2000-2025.csv")\n'

In [ ]:
# hitter = pd.read_csv("/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_batting_stats_by_season_1982-2025.csv")
# pitcher = pd.read_csv("/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_pitching_stats_by_season_1982-2025.csv")

hitter.head()
pitcher.head()

# 2015~2025년 데이터만 필터링
hitter_recent = hitter[hitter["year"] > 2015]
pitcher_recent = pitcher[pitcher["year"] > 2015]

hitter_recent.head()
pitcher_recent.head()

,순위,playerId,선수명,팀명,ERA,G,W,L,SV,HLD,...,SO,R,ER,WHIP,year,teamCode,teamName,CG,SHO,TBF
0,1,50904,이종준,LG,0.00,2,0,0,0,0,...,2,0,0,0.60,2025,LG,LG,NaN,NaN,NaN
1,2,77263,김강률,LG,1.46,12,1,0,1,4,...,9,2,2,1.22,2025,LG,LG,NaN,NaN,NaN
2,3,61145,이우찬,LG,1.89,23,0,1,0,0,...,20,6,4,1.42,2025,LG,LG,NaN,NaN,NaN
3,4,55167,김영우,LG,2.40,66,3,2,1,7,...,56,17,16,1.32,2025,LG,LG,NaN,NaN,NaN
4,5,50106,유영찬,LG,2.63,39,2,2,21,1,...,52,14,12,1.32,2025,LG,LG,NaN,NaN,NaN


### Github에 업로드 한 2026 raw 데이터 불러오기


In [ ]:
golden_glove = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_goldenglove_data_2025.csv")
hitter_2026 = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_hitter_data_2026.csv")
pitcher_2026 = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_pitcher_data_2026.csv")

golden_glove.head()
hitter_2026.head()
pitcher_2026.head()

,rank,name,team,ERA,G,W,L,SV,HLB,WPCT,...,NP,AVG,2B,3B,SAC,SF,IBB,WP,BK,year
0,1,후라도,삼성,2.61,12,3,1,0,0,0.750,...,1156,0.259,6,2,4,3,0,4,0,2026
1,2,올러,KIA,2.66,13,7,5,0,0,0.583,...,1211,0.182,5,2,5,1,0,3,0,2026
2,3,류현진,한화,2.84,12,8,2,0,0,0.800,...,1059,0.239,14,1,7,2,1,1,0,2026
3,4,최민석,두산,2.88,12,6,2,0,0,0.750,...,1098,0.215,13,0,3,2,0,3,1,2026
4,5,알칸타라,키움,3.12,12,6,4,0,0,0.600,...,1154,0.259,12,2,1,1,0,2,0,2026


## 데이터 전처리
### 1. Kaggle 데이터와 KBO 2026 데이터 칼럼명 맞추기

#### 기록용어 참고
01. 타자 기록
  - 2B: 2루타
  - 3B: 3루타
  - AB: 타수
  - AO: 뜬공
  - AVG: 타율
  - BB: 볼넷
  - BB/K: 볼넷/삼진
  - CS: 도루실패
  - E: 실책
  - G: 경기
  - GDP: 병살타
  - GO : 땅볼
  - GO/AO : 땅볼/뜬공
  - GPA : (1.8x출루율+장타율)/4
  - GW RBI : 결승타
  - H : 안타
  - HBP(HP) : 사구
  - HR : 홈런
  - IBB(IB) : 고의4구
  - ISOP : 순수장타율
  - MH : 멀티히트
  - OBP : 출루율
  - OPS : 출루율+장타율
  - P/PA : 투구수/타석
  - PA : 타석
  - PH-BA : 대타타율
  - R : 득점
  - RBI : 타점
  - RISP : 득점권타율
  - SAC(SH) : 희생번트
  - SB : 도루
  - SF : 희생플라이
  - SLG : 장타율
  - SO : 삼진
  - TB: 루타
  - XBH : 장타
  - XR : 추정득점

02. 투수기록
  - 2B : 2루타
  - 3B : 3루타
  - AO : 뜬공
  - AVG : 피안타율
  - BABIP : 인플레이타구타율
  - BB : 볼넷
  - BB/9 : 9이닝당 볼넷
  - BK : 보크
  - BSV : 블론세이브
  - CG : 완투
  - ER : 자책점
  - ERA : 평균자책점
  - G : 경기
  - GDP : 병살타
  - GF : 종료
  - GO : 땅볼
  - GO/AO : 땅볼/뜬공
  - GS : 선발
  - H : 피안타
  - HBP(HP) : 사구
  - HLD(HD) : 홀드
  - HR : 홈런
  - IBB(IB) : 고의4구
  - IP : 이닝
  - K/9 : 9이닝당 삼진
  - K/BB : 삼진/볼넷
  - L : 패
  - NP : 투구수
  - OBP : 피출루율
  - OPS : 피출루율+피장타율
  - P/G : 투구수/경기
  - P/IP : 투구수/이닝
  - QS : 퀄리티스타트
  - R : 실점
  - SAC(SH) : 희생번트
  - SF : 희생플라이
  - SHO : 완봉
  - SLG : 피장타율
  - SO : 삼진
  - SV(S) : 세이브
  - SVO : 세이브기회
  - TBF : 타자수
  - TS : 터프세이브
  - W : 승
  - Wgr : 구원승
  - Wgs : 선발승
  - WHIP : 이닝당 출루허용률
  - WP : 폭투
  - WPCT : 승률

####
   

In [ ]:
# 각 데이터의 칼럼명 확인
print(hitter_recent.columns)
print(pitcher_recent.columns)
print(hitter_2026.columns)
print(pitcher_2026.columns)
print(golden_glove.columns)


Index(['순위', 'playerId', '선수명', '팀명', 'AVG', 'G', 'PA', 'AB', 'R', 'H', '2B',
       '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'year', 'teamCode', 'teamName',
       'SB', 'CS', 'BB', 'HBP', 'SO', 'GDP', 'E'],
      dtype='object')
Index(['순위', 'playerId', '선수명', '팀명', 'ERA', 'G', 'W', 'L', 'SV', 'HLD',
       'WPCT', 'IP', 'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'year',
       'teamCode', 'teamName', 'CG', 'SHO', 'TBF'],
      dtype='object')
Index(['rank', 'name', 'team', 'position', 'AVG', 'G', 'PA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'BB', 'IBB', 'HBP', 'SO',
       'GDP', 'SLG', 'OBP', 'OPS', 'MH', 'RISP', 'PH-BA', 'year'],
      dtype='object')
Index(['rank', 'name', 'team', 'ERA', 'G', 'W', 'L', 'SV', 'HLB', 'WPCT', 'IP',
       'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'CG', 'SHO', 'QS',
       'BSV', 'TBF', 'NP', 'AVG', '2B', '3B', 'SAC', 'SF', 'IBB', 'WP', 'BK',
       'year'],
      dtype='object')
Index(['year', 'P', 'C', '1B', '2

In [ ]:
# 타자 데이터 컬럼명 정리(Kaggle 기준으로 정리)
"""
hitter_2026 = hitter_2026.rename(columns={
    "name": "Name",
    "team": "Team",
    "year": "Year",
    "position": "Pos",
    "HBP": "HP",
    "IBB": "IB",
    "SAC": "SH"
})
"""
# 2026 데이터 기준으로 정리
hitter_recent = hitter_recent.rename(columns = {
    "순위": "rank",
    "선수명": "name",
    "팀명": "team",
})

pitcher_recent = pitcher_recent.rename(columns = {
    "순위": "rank",
    "선수명": "name",
    "팀명": "team",
})

# Strip whitespace from name and team columns
hitter_recent["name"] = hitter_recent["name"].str.strip()
hitter_recent["team"] = hitter_recent["team"].str.strip()
pitcher_recent["name"] = pitcher_recent["name"].str.strip()
pitcher_recent["team"] = pitcher_recent["team"].str.strip()

# 투수 데이터 컬럼명 정리(Kaggle 기준으로 정리)
"""
pitcher_2026 = pitcher_2026.rename(columns={
    "name": "Name",
    "team": "Team",
    "year": "Year",
    "HLB": "HD",
    "SV": "S"
})
"""

print(hitter_2026.columns)
print(pitcher_2026.columns)
print(hitter_recent.columns)
print(pitcher_recent.columns)

Index(['rank', 'name', 'team', 'position', 'AVG', 'G', 'PA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'BB', 'IBB', 'HBP', 'SO',
       'GDP', 'SLG', 'OBP', 'OPS', 'MH', 'RISP', 'PH-BA', 'year'],
      dtype='object')
Index(['rank', 'name', 'team', 'ERA', 'G', 'W', 'L', 'SV', 'HLB', 'WPCT', 'IP',
       'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'CG', 'SHO', 'QS',
       'BSV', 'TBF', 'NP', 'AVG', '2B', '3B', 'SAC', 'SF', 'IBB', 'WP', 'BK',
       'year'],
      dtype='object')
Index(['rank', 'playerId', 'name', 'team', 'AVG', 'G', 'PA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'year', 'teamCode',
       'teamName', 'SB', 'CS', 'BB', 'HBP', 'SO', 'GDP', 'E'],
      dtype='object')
Index(['rank', 'playerId', 'name', 'team', 'ERA', 'G', 'W', 'L', 'SV', 'HLD',
       'WPCT', 'IP', 'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'year',
       'teamCode', 'teamName', 'CG', 'SHO', 'TBF'],
      dtype='object')


### 2. 골든글러브 데이터 전처리
#### 2-1. 칼럼명 정리
year, P, C, 1B, 2B, 3B, SS, OF1, OF2, OF3, OF4, DH(기존) --> year, position, player_team(전처리 후)

#### 2-2. 선수명/팀명 분리
현재 데이터에는 `폰세 한화`로 되어있어서 분리가 필요하다.

#### 2-3. 2015-2025 학습 데이터에 `is_goldenglove` 라벨 추가

In [ ]:
# 2-1. 칼럼 및 데이터 정리
position_cols = ["P", "C", "1B", "2B", "3B", "SS", "OF1", "OF2", "OF3", "OF4", "DH"]

golden = golden_glove.melt(
    id_vars = "year",
    value_vars = position_cols,
    var_name = "position",
    value_name = "player_name"
)

# 빈 값 제거
golden = golden.dropna(subset=["player_name"])

# OF1..4 -> OF로 통일
golden["position"] = golden["position"].replace({
    "OF1": "OF",
    "OF2": "OF",
    "OF3": "OF",
    "OF4": "OF"
})

# 선수명, 팀이름 분리
golden[["name", 'team']] = golden["player_name"].str.rsplit(" ", n = 1, expand = True)

# Strip whitespace after splitting
golden["name"] = golden["name"].str.strip()
golden["team"] = golden["team"].str.strip()

golden = golden[["year", "position", "name", "team"]]

# Rename columns to match Kaggle data's capitalization for merging/matching later
golden = golden.rename(columns={"year": "Year", "name": "Name", "team": "Team"})

golden.head()

,Year,position,Name,Team
0,2025,P,폰세,한화
1,2024,P,하트,NC
2,2023,P,페디,NC
3,2022,P,안우진,키움
4,2021,P,미란다,두산


In [ ]:
# 학습용 kaggle 데이터에 is_goldenglove 라벨 붙이기
hitter_recent["is_goldenglove"] = hitter_recent.set_index(["year", "name", "team"]).index.isin(
    golden.set_index(["Year", "Name", "Team"]).index
).astype(int)

pitcher_recent["is_goldenglove"] = pitcher_recent.set_index(["year", "name", "team"]).index.isin(
  golden.set_index(["Year", "Name", "Team"]).index
).astype(int)

hitter_recent.head()
pitcher_recent.head()

# is_goldenglove 라벨 확인: 골든글러브 수상자면 1, 아니면 0
hitter_recent[hitter_recent["name"] == "김도영"].head()


,rank,playerId,name,team,AVG,G,PA,AB,R,H,...,teamCode,teamName,SB,CS,BB,HBP,SO,GDP,E,is_goldenglove
272,3,52605,김도영,KIA,0.309,30,122,110,20,34,...,HT,KIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
400,3,52605,김도영,KIA,0.347,141,625,544,143,189,...,HT,KIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
964,4,52605,김도영,KIA,0.303,84,385,340,72,103,...,HT,KIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1311,16,52605,김도영,KIA,0.237,103,254,224,37,53,...,HT,KIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [ ]:
# 라벨이 잘 붙었는지 확인

print(hitter_recent["is_goldenglove"].value_counts())
print(pitcher_recent["is_goldenglove"].value_counts())

print(hitter_recent.groupby("year")["is_goldenglove"].sum())
print(pitcher_recent.groupby("year")["is_goldenglove"].sum())

is_goldenglove
0    3610
1      85
Name: count, dtype: int64
is_goldenglove
0    2724
1      11
Name: count, dtype: int64
year
2016    8
2017    8
2018    8
2019    9
2020    9
2021    9
2022    8
2023    9
2024    9
2025    8
Name: is_goldenglove, dtype: int64
year
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    2
2023    1
2024    1
2025    1
Name: is_goldenglove, dtype: int64


In [ ]:
# 팀 이적으로 인한 골든글러브 라벨 생성 실패 건 확인
# 골든글러브 수상자 팀명 보정
# 기준: 시상식 당시 소속이 아니라 해당 시즌 정규시즌 기록 데이터의 Team 값

team_corrections = {
    (2017, "강민호"): "롯데",
    (2016, "최형우"): "삼성",
    (2022, "양의지"): "NC",
    (2025, "최형우"): "KIA"
}

for (year, name), team in team_corrections.items():
    golden.loc[
        (golden["Year"] == year) & (golden["Name"] == name),
        "Team"
    ] = team

# 2018 이정후 선수의 Name, Team 분리 안되는 문제 해결
golden.loc[
    (golden["Year"] == 2018) &
    (golden["Team"].str.contains("이정후", na=False)),
    ["Name", "Team"]
] = ["이정후", "넥센"]


In [ ]:
# golden_hitter_recent 생성 후 매칭 확인
hitter_positions = ["C", "1B", "2B", "3B", "SS", "OF", "DH"]

golden_hitter_recent = golden[
    (golden["position"].isin(hitter_positions)) &
    (golden["Year"] >= 2016) &
    (golden["Year"] <= 2025)
].copy()

hitter_keys = set(zip(hitter_recent["year"], hitter_recent["name"], hitter_recent["team"]))

golden_hitter_recent["matched"] = golden_hitter_recent.apply(
    lambda row: (row["Year"], row["Name"], row["Team"]) in hitter_keys,
    axis=1
)

failed_hitter = golden_hitter_recent[golden_hitter_recent["matched"] == False]

print("매칭 실패 인원:", len(failed_hitter))
display(failed_hitter)

print(golden["Year"].dtype)
print(hitter_recent["year"].dtype)

print(golden["Name"].dtype)
print(hitter_recent["name"].dtype)

print(golden["Team"].dtype)
print(hitter_recent["team"].dtype)

매칭 실패 인원: 1


,Year,position,Name,Team,matched
189,2018,OF,이정후 넥센,,False


int64
int64
object
object
object
object


In [ ]:
# 라벨 여부 한번 더 확인

pd.set_option("display.max_rows", None)      # 행 전체 출력
pd.set_option("display.max_columns", None)   # 열 전체 출력
pd.set_option("display.width", None)         # 줄바꿈 제한 완화
pd.set_option("display.max_colwidth", None)  # 셀 내용 전체 출력

hitter_recent[hitter_recent["is_goldenglove"] == 1]
pitcher_recent[pitcher_recent["is_goldenglove"] == 1]

,rank,playerId,name,team,ERA,G,W,L,SV,HLD,WPCT,IP,H,HR,BB,HBP,SO,R,ER,WHIP,year,teamCode,teamName,CG,SHO,TBF,is_goldenglove
29,1,55730,폰세,한화,1.89,29,17,1,0,0,0.944,180 2/3,128,10,41,6,252,41,38,0.94,2025,HH,한화,NaN,NaN,NaN,1
513,2,54930,하트,NC,2.69,26,13,3,0,0,0.813,157,124,11,38,11,182,51,47,1.03,2024,NC,NC,NaN,NaN,NaN,1
655,3,53913,페디,NC,2.00,30,20,6,0,0,0.769,180 1/3,137,9,35,4,209,46,40,0.95,2023,NC,NC,NaN,NaN,NaN,1
888,3,68341,안우진,키움,2.11,30,15,8,0,0,0.652,196,131,4,55,4,224,51,46,0.95,2022,WO,키움,NaN,NaN,NaN,1
1049,1,71564,이대호,롯데,0.00,1,0,0,0,1,-,1/3,0,0,0,0,0,0,0,0.00,2022,LT,롯데,NaN,NaN,NaN,1
1167,6,51257,미란다,두산,2.33,28,14,5,0,0,0.737,173 2/3,135,11,63,1,225,49,45,1.14,2021,OB,두산,NaN,NaN,NaN,1
1480,3,69045,알칸타라,두산,2.54,31,20,2,0,0,0.909,198 2/3,174,12,30,9,182,58,56,1.03,2020,OB,두산,NaN,NaN,NaN,1
1732,4,65543,린드블럼,두산,2.50,30,20,3,0,0,0.870,194 2/3,165,13,29,6,189,57,54,1.00,2019,OB,두산,NaN,NaN,NaN,1
2014,1,65543,린드블럼,두산,2.88,26,15,4,0,0,0.789,168 2/3,142,16,38,8,157,56,54,1.07,2018,OB,두산,NaN,NaN,NaN,1
2248,3,77637,양현종,KIA,3.44,31,20,6,0,0,0.769,193 1/3,209,17,45,0,158,88,74,1.31,2017,HT,KIA,NaN,NaN,NaN,1


In [ ]:
# is_goldenglove 라벨 재생성

# 2018 이정후 매칭 보정
# golden에는 2018 이정후가 있는데 매칭이 안 되는 문제를 수동 보정

golden.loc[
    (golden["Year"] == 2018) &
    (golden["Name"].astype(str).str.contains("이정후", na=False)),
    ["Name", "Team"]
] = ["이정후", "넥센"]

# 1. 타입/문자열 정리
golden["Year"] = pd.to_numeric(golden["Year"], errors="coerce").astype("Int64")
hitter_recent["year"] = pd.to_numeric(hitter_recent["year"], errors="coerce").astype("Int64")
pitcher_recent["year"] = pd.to_numeric(pitcher_recent["year"], errors="coerce").astype("Int64")

for col in ["Name", "Team"]:
    golden[col] = (
        golden[col]
        .astype(str)
        .str.replace("\xa0", " ", regex=False)
        .str.replace("\u200b", "", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

for col in ["name", "team"]:
    hitter_recent[col] = (
        hitter_recent[col]
        .astype(str)
        .str.replace("\xa0", " ", regex=False)
        .str.replace("\u200b", "", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    pitcher_recent[col] = (
        pitcher_recent[col]
        .astype(str)
        .str.replace("\xa0", " ", regex=False)
        .str.replace("\u200b", "", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# 2. 타자/투수 골든글러브 정답 목록 분리
hitter_positions = ["C", "1B", "2B", "3B", "SS", "OF", "DH"]

golden_hitter_recent = golden[
    (golden["position"].isin(hitter_positions)) &
    (golden["Year"] >= 2016) &
    (golden["Year"] <= 2025)
].copy()

golden_pitcher_recent = golden[
    (golden["position"] == "P") &
    (golden["Year"] >= 2000) &
    (golden["Year"] <= 2025)
].copy()

# 3. 중복 제거
golden_hitter_recent = golden_hitter_recent.drop_duplicates(
    subset=["Year", "Name", "Team", "position"]
)

golden_pitcher_recent = golden_pitcher_recent.drop_duplicates(
    subset=["Year", "Name", "Team"]
)

hitter_recent = hitter_recent.drop_duplicates(
    subset=["year", "name", "team"],
    keep="first"
).copy()

pitcher_recent = pitcher_recent.drop_duplicates(
    subset=["year", "name", "team"],
    keep="first"
).copy()

# 4. 라벨 다시 생성
hitter_recent["is_goldenglove"] = hitter_recent.set_index(
    ["year", "name", "team"]
).index.isin(
    golden_hitter_recent.set_index(["Year", "Name", "Team"]).index
).astype(int)

pitcher_recent["is_goldenglove"] = pitcher_recent.set_index(
    ["year", "name", "team"]
).index.isin(
    golden_pitcher_recent.set_index(["Year", "Name", "Team"]).index
).astype(int)

# 5. 결과 확인
print("타자 연도별 골든글러브 라벨 수")
print(hitter_recent.groupby("year")["is_goldenglove"].sum())

print("\n투수 연도별 골든글러브 라벨 수")
print(pitcher_recent.groupby("year")["is_goldenglove"].sum())

print("\n타자 라벨 분포")
print(hitter_recent["is_goldenglove"].value_counts())

print("\n투수 라벨 분포")
print(pitcher_recent["is_goldenglove"].value_counts())

타자 연도별 골든글러브 라벨 수
year
2016    9
2017    9
2018    9
2019    9
2020    9
2021    9
2022    9
2023    9
2024    9
2025    9
Name: is_goldenglove, dtype: int64

투수 연도별 골든글러브 라벨 수
year
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    1
2023    1
2024    1
2025    1
Name: is_goldenglove, dtype: int64

타자 라벨 분포
is_goldenglove
0    3598
1      90
Name: count, dtype: int64

투수 라벨 분포
is_goldenglove
0    2719
1      10
Name: count, dtype: int64


In [ ]:
# 타자 결측치 비율 확인(NaN)
hitter_missing = hitter_recent.isna().mean().sort_values(ascending=False)
print("타자 결측치 비율")
display(hitter_missing[hitter_missing > 0])

# playerId, teamName, teamCode 칼럼 삭제
# 타자 데이터 SO, GDP, E, HBP, BB, SB, CS 결측치 처리 필요

타자 결측치 비율


,0
SO,1.0
GDP,1.0
E,1.0
HBP,1.0
BB,1.0
SB,1.0
CS,1.0


In [ ]:
# 투수 결측치 비율 확인(NaN)
pitcher_missing = pitcher_recent.isna().mean().sort_values(ascending=False)
print("투수 결측치 비율")
display(pitcher_missing[pitcher_missing > 0])

투수 결측치 비율


,0
SHO,1.00000000
CG,1.00000000
TBF,1.00000000


In [ ]:
import numpy as np

# 학습 데이터의 불필요한 공통 칼럼 제거

# 삭제하면 안되는 칼럼
protected_cols_hitter = ["year", "name", "team", "position", "is_goldenglove"]
protected_cols_pitcher = ["year", "name", "team", "is_goldenglove"]

# 공통 삭제 칼럼
extra_drop_cols = ["playerId", "teamCode", "teamName"]

# 원본 copy
hitter_recent_clean = hitter_recent.copy()
pitcher_recent_clean = pitcher_recent.copy()

# .isna()는 "-"를 결측치로 보지 않기때문에 NaN으로 변
hitter_recent_clean = hitter_recent_clean.replace("-", np.nan)
pitcher_recent_clean = pitcher_recent_clean.replace("-", np.nan)

# 결측치 50% 이상 칼럼 찾기
hitter_missing_ratio = hitter_recent_clean.isna().mean()
pitcher_missing_ratio = pitcher_recent_clean.isna().mean()

hitter_drop_cols = [
    col for col in hitter_missing_ratio[hitter_missing_ratio >= 0.5].index
    if col not in protected_cols_hitter
]

pitcher_drop_cols = [
    col for col in pitcher_missing_ratio[pitcher_missing_ratio >= 0.5].index
    if col not in protected_cols_pitcher
]

# 불필요한 칼럼 제거
hitter_drop_cols.extend([
    col for col in extra_drop_cols
    if col in hitter_recent_clean.columns and col not in hitter_drop_cols
])

pitcher_drop_cols.extend([
    col for col in extra_drop_cols
    if col in pitcher_recent_clean.columns and col not in pitcher_drop_cols
])

print("삭제할 타자 컬럼:", hitter_drop_cols)
print("삭제할 투수 컬럼:", pitcher_drop_cols)

# 컬럼 삭제
hitter_recent_clean = hitter_recent_clean.drop(columns=hitter_drop_cols).copy()
pitcher_recent_clean = pitcher_recent_clean.drop(columns=pitcher_drop_cols).copy()

삭제할 타자 컬럼: ['SB', 'CS', 'BB', 'HBP', 'SO', 'GDP', 'E', 'playerId', 'teamCode', 'teamName']
삭제할 투수 컬럼: ['CG', 'SHO', 'TBF', 'playerId', 'teamCode', 'teamName']


/tmp/ipykernel_2471/3257952003.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  hitter_recent_clean = hitter_recent_clean.replace("-", np.nan)
/tmp/ipykernel_2471/3257952003.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pitcher_recent_clean = pitcher_recent_clean.replace("-", np.nan)


In [ ]:
# 결측치 칼럼 및 공통 칼럽 삭제 후 데이터프레임의 결측치 비율 확인
print("\n--- hitter_recent Missing Values ---")
print(hitter_recent_clean.isna().mean().sort_values(ascending=False).head(30))

print("\n--- pitcher_recent Missing Values ---")
print(pitcher_recent_clean.isna().mean().sort_values(ascending=False).head(30))


--- hitter_recent Missing Values ---
AVG              0.22180043
rank             0.00000000
name             0.00000000
team             0.00000000
G                0.00000000
PA               0.00000000
AB               0.00000000
R                0.00000000
H                0.00000000
2B               0.00000000
3B               0.00000000
HR               0.00000000
TB               0.00000000
RBI              0.00000000
SAC              0.00000000
SF               0.00000000
year             0.00000000
is_goldenglove   0.00000000
dtype: float64

--- pitcher_recent Missing Values ---
WPCT             0.27445951
WHIP             0.00366435
ERA              0.00366435
rank             0.00000000
name             0.00000000
G                0.00000000
team             0.00000000
SV               0.00000000
W                0.00000000
HLD              0.00000000
IP               0.00000000
H                0.00000000
L                0.00000000
HR               0.00000000
BB          

## 모델 학습
```
1. 지도학습 모델
   - 로지스틱 회귀
   - 의사결정트리
   - 퍼셉트론
   → is_goldenglove = 1/0 예측

2. 비지도학습 분석
   - K-Means 군집화
   → 선수 기록 특성에 따라 선수 유형 분류
```
군집화 분석은 골든글러브 수상 여부를 직접 예측하기 위한 모델이 아니라, 선수들의 기록 패턴을 기준으로 유사한 선수 유형을 파악하기 위한 보조 분석으로 활용하였다. 이를 통해 예측 확률이 높은 선수들이 어떤 기록 특성을 가진 군집에 속하는지 함께 해석하였다.   

### 1. 타자 모델 학습 결과
모델 학습후 Accuracy를 확인했을 때
| 모델      | Accuracy | class 1 Precision | class 1 Recall | 해석                 |
| ------- | -------: | ----------------: | -------------: | ------------------ |
| 로지스틱 회귀 |    0.927 |              0.21 |           0.72 | 후보 예측에 균형적         |
| 의사결정트리  |    0.939 |              0.22 |           0.61 | 설명력 좋음, 정확도 높음     |
| 퍼셉트론    |    0.905 |              0.20 |           1.00 | 수상자를 놓치지 않지만 과다 예측 |

골든글러브 수상자는 전체 선수 중 매우 적기 때문에 데이터 불균형이 크다. 따라서 단순 정확도보다 수상자 클래스(1)에 대한 recall과 f1-score를 함께 고려하였다. 의사결정트리는 가장 높은 정확도를 보였으나, 로지스틱 회귀는 수상자 클래스에 대해 비교적 높은 recall을 보여 후보군 예측에 적합하였다. 퍼셉트론은 수상자 recall이 1.00으로 가장 높았지만 precision이 낮아 수상 가능성이 있는 선수를 넓게 잡는 경향을 보였다.   

타자 골든글러브 예측 모델 평가 결과, 세 모델 모두 90% 이상의 정확도를 보였다. 그러나 골든글러브 수상자는 전체 선수 중 소수이므로 단순 정확도만으로 모델 성능을 판단하기에는 한계가 있다. 수상자 클래스(1)를 기준으로 보면 로지스틱 회귀는 recall 0.72, f1-score 0.33을 보였고, 의사결정트리는 recall 0.61, f1-score 0.33을 보였다. 퍼셉트론은 recall 1.00으로 실제 수상자를 모두 포착했지만 precision이 0.20으로 낮아 수상 후보를 과도하게 넓게 예측하는 경향을 보였다. 따라서 본 분석에서는 예측 확률 해석이 가능하고 후보군 예측에 비교적 적합한 로지스틱 회귀를 중심으로 2026년 골든글러브 후보를 예측하기로 결정했다.

### 2. 투수 모델 학습 결과
| 모델      | Accuracy | class 1 Precision | class 1 Recall | 해석                      |
| ------- | -------: | ----------------: | -------------: | ----------------------- |
| 로지스틱 회귀 |    0.996 |              0.00 |           0.00 | 실제 수상자를 잡지 못함           |
| 의사결정트리  |    0.996 |              0.50 |           0.50 | 테스트셋 기준 가장 양호하지만 표본이 작음 |
| 퍼셉트론    |    0.842 |              0.02 |           1.00 | 수상자를 모두 잡지만 과다 예측       |

투수 부문은 학습 기간 내 골든글러브 수상자가 10명뿐이므로, 모델이 수상자 클래스의 패턴을 충분히 학습하기 어려웠다. 따라서 투수 예측 결과는 최종 수상자를 단정하기보다, 현재 성적 기준으로 수상 가능성이 높은 후보군을 정렬하는 방식으로 해석하였다.   

투수 골든글러브 수상자는 연도별 1명으로 타자 부문보다 클래스(1)의 수가 매우 적다. 이로 인해 train/test 분할 시 테스트 데이터에 포함되는 실제 수상자 수가 적어 precision, recall, f1-score가 불안정하게 나타났다. 따라서 투수 모델의 평가지표는 참고용으로 해석하고, 최종 예측에서는 로지스틱 회귀의 예측 확률을 기준으로 후보군을 순위화하여 결정하기로 했다.   

투수의 경우 분류 정확도 보단 확률 순위로 `2026년 6월 14일 기준 기록상 수상 가능성이 높은 투수 순위`를 확인하는 것이 맞는 방향으로 판단했다.

In [ ]:
# 1. feature 만들기

# 타자 feature
hitter_features = [
    "AVG", "G", "PA", "AB", "R", "H",
    "2B", "3B", "HR", "TB", "RBI", "SAC", "SF"
]

# 실제 존재하는 칼럼만 사용
hitter_features = [col for col in hitter_features
                   if col in hitter_recent_clean.columns]

# 투수 feature
pitcher_features = [
    "ERA", "G", "W", "L", "SV", "HLD", "HD", "IP",
    "H", "HR", "BB", "HBP", "SO", "R", "ER", "WHIP"
]

pitcher_features = [col for col in pitcher_features
                    if col in pitcher_recent_clean.columns]

# 수치형 변환
for col in hitter_features:
    hitter_recent_clean[col] = pd.to_numeric(hitter_recent_clean[col], errors="coerce")

for col in pitcher_features:
    pitcher_recent_clean[col] = pd.to_numeric(pitcher_recent_clean[col], errors="coerce")

# 남은 결측치 중앙값 대체
for col in hitter_features: # Changed: Added this loop
    median_value = hitter_recent_clean[col].median()
    hitter_recent_clean[col] = hitter_recent_clean[col].fillna(median_value)

for col in pitcher_features:
    median_value = pitcher_recent_clean[col].median()
    pitcher_recent_clean[col] = pitcher_recent_clean[col].fillna(median_value)


print("\n최종 타자 feature:")
print(hitter_features)

print("\n최종 투수 feature:")
print(pitcher_features)

print("\n타자 feature 결측치 확인:")
print(hitter_recent_clean[hitter_features].isna().sum())

print("\n투수 feature 결측치 확인:")
print(pitcher_recent_clean[pitcher_features].isna().sum())

print("\n타자 데이터 타입:")
print(hitter_recent_clean[hitter_features].dtypes)

print("\n투수 데이터 타입:")
print(pitcher_recent_clean[pitcher_features].dtypes)

print("\n전처리 완료")
print("타자 데이터 크기:", hitter_recent_clean.shape)
print("투수 데이터 크기:", pitcher_recent_clean.shape)


최종 타자 feature:
['AVG', 'G', 'PA', 'AB', 'R', 'H', '2B', '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF']

최종 투수 feature:
['ERA', 'G', 'W', 'L', 'SV', 'HLD', 'IP', 'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP']

타자 feature 결측치 확인:
AVG    0
G      0
PA     0
AB     0
R      0
H      0
2B     0
3B     0
HR     0
TB     0
RBI    0
SAC    0
SF     0
dtype: int64

투수 feature 결측치 확인:
ERA     0
G       0
W       0
L       0
SV      0
HLD     0
IP      0
H       0
HR      0
BB      0
HBP     0
SO      0
R       0
ER      0
WHIP    0
dtype: int64

타자 데이터 타입:
AVG    float64
G        int64
PA       int64
AB       int64
R        int64
H        int64
2B       int64
3B       int64
HR       int64
TB       int64
RBI      int64
SAC      int64
SF       int64
dtype: object

투수 데이터 타입:
ERA     float64
G         int64
W         int64
L         int64
SV        int64
HLD       int64
IP      float64
H         int64
HR        int64
BB        int64
HBP       int64
SO        int64
R         int64
ER        int64
WHIP    

In [ ]:
# 학습 데이터 분리
x_hitter = hitter_recent_clean[hitter_features]
y_hitter = hitter_recent_clean["is_goldenglove"]

x_pitcher = pitcher_recent_clean[pitcher_features]
y_pitcher = pitcher_recent_clean["is_goldenglove"]


In [ ]:
# 모델 학습 사용 모델 및 의존성 설정
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
import numpy as np

# 타자 모델 학습
X_train, X_test, y_train, y_test = train_test_split(
    x_hitter,
    y_hitter,
    test_size = 0.2,
    random_state = 42,
    stratify = y_hitter
)

hitter_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter = 1000, class_weight = "balanced"))
    ]),
    "Decision Tree" : Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier (
            max_depth = 4,
            random_state = 42,
            class_weight = "balanced"
        ))
    ]),
    "Perceptron": Pipeline([
      ("imputer", SimpleImputer(strategy="median")),
      ("scaler", StandardScaler()),
      ("model", Perceptron(
          max_iter = 1000,
          random_state = 42,
          class_weight = "balanced"
      ))
    ])
}

for name, model in hitter_models.items():
  model.fit(X_train, y_train)
  pred = model.predict(X_test)

  print("===============")
  print(name)
  print("Accuracy: ", accuracy_score(y_test, pred))
  print(classification_report(y_test, pred))

Logistic Regression
Accuracy:  0.926829268292683
              precision    recall  f1-score   support

           0       0.99      0.93      0.96       720
           1       0.21      0.72      0.33        18

    accuracy                           0.93       738
   macro avg       0.60      0.83      0.64       738
weighted avg       0.97      0.93      0.95       738

Decision Tree
Accuracy:  0.9390243902439024
              precision    recall  f1-score   support

           0       0.99      0.95      0.97       720
           1       0.22      0.61      0.33        18

    accuracy                           0.94       738
   macro avg       0.61      0.78      0.65       738
weighted avg       0.97      0.94      0.95       738

Perceptron
Accuracy:  0.9051490514905149
              precision    recall  f1-score   support

           0       1.00      0.90      0.95       720
           1       0.20      1.00      0.34        18

    accuracy                           0.91     

In [ ]:
# 투수 모델 학습
X_train_pitcher, X_test_pitcher, y_train_pitcher, y_test_pitcher = train_test_split(
    x_pitcher,
    y_pitcher,
    test_size = 0.2,
    random_state = 42,
    stratify = y_pitcher
)

pitcher_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter = 1000, class_weight = "balanced"))
    ]),
    "Decision Tree" : Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier (
            max_depth = 4,
            random_state = 42,
            class_weight = "balanced"
        ))
    ]),
    "Perceptron": Pipeline([
      ("imputer", SimpleImputer(strategy="median")),
      ("scaler", StandardScaler()),
      ("model", Perceptron(
          max_iter = 1000,
          random_state = 42,
          class_weight = "balanced"
      ))
    ])
}

for name, model in pitcher_models.items():
  model.fit(X_train_pitcher, y_train_pitcher)
  pred = model.predict(X_test_pitcher)

  print("===============")
  print(name)
  print("Accuracy: ", accuracy_score(y_test_pitcher, pred))
  print(classification_report(y_test_pitcher, pred))

Logistic Regression
Accuracy:  0.9963369963369964
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       544
           1       0.00      0.00      0.00         2

    accuracy                           1.00       546
   macro avg       0.50      0.50      0.50       546
weighted avg       0.99      1.00      0.99       546

Decision Tree
Accuracy:  0.9963369963369964
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       544
           1       0.50      0.50      0.50         2

    accuracy                           1.00       546
   macro avg       0.75      0.75      0.75       546
weighted avg       1.00      1.00      1.00       546

Perceptron
Accuracy:  0.8424908424908425
              precision    recall  f1-score   support

           0       1.00      0.84      0.91       544
           1       0.02      1.00      0.04         2

    accuracy                           0.84    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# 타자 최종 모델: 로지스틱 회귀
# 투수 최종 모델: 의사결정트리
hitter_final_model = hitter_models["Logistic Regression"]
hitter_final_model.fit(x_hitter, y_hitter)

pitcher_final_model = pitcher_models["Logistic Regression"]
pitcher_final_model.fit(x_pitcher, y_pitcher)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

## 2026 골든글러브 예측
데이터는 2026-06-14일 데이터를 기준으로 확인한다.   

본 분석에서는 골든글러브 수상 여부를 0과 1로 구분하여 지도학습 분류 모델을 학습하였다. 이후 2026년 선수 데이터에는 단순 예측값이 아니라 predict_proba()를 활용해 수상 가능성 점수를 산출하였다. 골든글러브는 포지션별로 수상자가 정해지는 특성이 있으므로, 타자는 각 포지션별 예측 확률 상위 선수를 후보로 선정하였고, 투수는 전체 투수 중 예측 확률 상위 선수를 후보로 선정하였다. 특히 투수 부문은 수상자 라벨 수가 적어 확률값이 매우 낮게 산출되었기 때문에, 확률의 절댓값보다 후보 간 상대 순위를 중심으로 해석하였다.   

지명타자(DH)는 골든글러브 수상 부문에는 포함되지만, 본 분석에 사용한 2026년 타자 데이터의 포지션 컬럼에는 DH가 별도 포지션으로 제공되지 않았다. 따라서 2026년 포지션별 후보 선정 과정에서는 데이터상 확인 가능한 수비 포지션(C, 1B, 2B, 3B, SS, OF)을 기준으로 후보를 선정하였으며, DH 부문은 데이터 한계로 인해 별도 자동 선정에서 제외하였다.

In [ ]:
# 2026 타자 데이터 예측

# 원본 복사
hitter_2026_model = hitter_2026.copy()

# 학습 때 사용한 feature 기준으로 숫자형 변환
for col in hitter_features:
  if col in hitter_2026_model.columns:
    hitter_2026_model[col] = pd.to_numeric(
        hitter_2026_model[col],
        errors="coerce"
    )

# 학습 때 사용한 feature만 추출
x_hitter_2026 = hitter_2026_model[hitter_features]

# 예측 확률 생성
hitter_2026_model["goldenglove_probability"] = (
    hitter_final_model.predict_proba(x_hitter_2026)[:, 1]
)

# 확률 높은 순으로 정렬
hitter_result = hitter_2026_model.sort_values(
    "goldenglove_probability",
    ascending = False
)

hitter_result[[
    "year", "name", "team", "position",
    "AVG", "G", "PA", "AB", "R", "H", "HR", "RBI", "goldenglove_probability"
]].head(30)

,year,name,team,position,AVG,G,PA,AB,R,H,HR,RBI,goldenglove_probability
4,2026,오스틴,LG,1B,0.34900000,63,286,252,53,88,19,59,0.21247036
0,2026,최원준,KT,OF,0.38600000,62,297,254,54,98,5,37,0.14406165
2,2026,박성한,SSG,SS,0.35500000,63,280,231,46,82,3,32,0.07736969
5,2026,페라자,한화,OF,0.32900000,62,287,240,54,79,13,40,0.07539780
3,2026,레이예스,롯데,OF,0.35200000,62,277,247,28,87,9,44,0.06442936
32,2026,힐리어드,KT,OF,0.27100000,63,280,247,47,67,13,48,0.04027328
9,2026,박건우,NC,OF,0.30900000,60,241,204,32,63,12,34,0.03708608
30,2026,김도영,KIA,3B,0.27200000,64,276,235,45,64,19,52,0.03517715
6,2026,강백호,한화,1B,0.32300000,58,256,226,37,73,13,63,0.03256187
17,2026,문현빈,한화,OF,0.29200000,60,280,236,41,69,9,44,0.03076620


In [ ]:
# 2026 투수 데이터 예측

# 원본 복사
pitcher_2026_model = pitcher_2026.copy()

# 'HLB' 컬럼을 'HLD'로 변경 (feature list와 일치시키기 위해)
if 'HLB' in pitcher_2026_model.columns:
    pitcher_2026_model = pitcher_2026_model.rename(columns={'HLB': 'HLD'})

# 학습 때 사용한 feature 기준으로 숫자형 변환
for col in pitcher_features:
  if col in pitcher_2026_model.columns:
    pitcher_2026_model[col] = pd.to_numeric(
        pitcher_2026_model[col],
        errors="coerce"
    )

# 학습 때 사용한 feature만 추출
x_pitcher_2026 = pitcher_2026_model[pitcher_features]

# 예측 확률 생성
pitcher_2026_model["goldenglove_probability"] = (
    pitcher_final_model.predict_proba(x_pitcher_2026)[:, 1]
)

# 확률 높은 순으로 정렬
pitcher_result = pitcher_2026_model.sort_values(
    "goldenglove_probability",
    ascending = False
)

pitcher_result[[
    "year", "name", "team",
    "ERA", "G", "W", "L", "IP", "H", "HR", "BB",
    "SO", "WHIP",
    "goldenglove_probability"
]].head(30)

,year,name,team,ERA,G,W,L,IP,H,HR,BB,SO,WHIP,goldenglove_probability
2,2026,류현진,한화,2.84000000,12,8,2,NaN,62,4,10,56,1.03000000,0.00181691
1,2026,올러,KIA,2.66000000,13,7,5,NaN,52,5,25,86,0.95000000,0.00078523
4,2026,알칸타라,키움,3.12000000,12,6,4,78.00000000,79,11,11,70,1.15000000,0.00075618
9,2026,구창모,NC,3.69000000,12,6,2,NaN,64,8,21,56,1.24000000,0.00023976
3,2026,최민석,두산,2.88000000,12,6,2,NaN,54,3,31,67,1.24000000,0.00020995
12,2026,톨허스트,LG,3.99000000,13,7,4,70.00000000,65,6,19,59,1.20000000,0.00012316
6,2026,곽빈,두산,3.38000000,12,4,3,NaN,68,4,24,79,1.38000000,0.00010997
16,2026,고영표,KT,4.50000000,12,4,4,68.00000000,74,8,11,78,1.25000000,0.00009682
10,2026,임찬규,LG,3.72000000,12,6,1,NaN,78,5,23,36,1.55000000,0.00005700
0,2026,후라도,삼성,2.61000000,12,3,1,76.00000000,76,8,13,50,1.17000000,0.00005339


In [ ]:
# 타자 최종후보

# 타자 예측 확률 기준 정렬
hitter_result = hitter_2026_model.sort_values(
    "goldenglove_probability",
    ascending=False
)

selected_hitters = []

for pos in ["C", "1B", "2B", "3B", "SS", "DH"]:
    temp = hitter_result[hitter_result["position"] == pos].head(1)
    if len(temp) > 0:
        selected_hitters.append(temp)

of_top3 = hitter_result[hitter_result["position"] == "OF"].head(3)
if len(of_top3) > 0:
    selected_hitters.append(of_top3)

final_hitter_candidates = pd.concat(selected_hitters)

print("타자 최종 후보: ")
final_hitter_candidates.head(30)

타자 최종 후보: 


,rank,name,team,position,AVG,G,PA,AB,R,H,2B,3B,HR,TB,RBI,SAC,SF,BB,IBB,HBP,SO,GDP,SLG,OBP,OPS,MH,RISP,PH-BA,year,goldenglove_probability
40,41,양의지,두산,C,0.25600000,61,246,207,22,53,7,0,10,90,34,0,4,22,0,13,31,5,0.43500000,0.35800000,0.79300000,14,0.22200000,0.50000000,2026,0.00847196
4,5,오스틴,LG,1B,0.34900000,63,286,252,53,88,15,3,19,166,59,0,2,28,2,4,41,5,0.65900000,0.42000000,1.07900000,22,0.41100000,0.00000000,2026,0.21247036
10,11,박민우,NC,2B,0.30600000,60,251,206,39,63,13,0,4,88,37,1,2,36,2,6,30,6,0.42700000,0.42000000,0.84700000,16,0.42400000,0.50000000,2026,0.01610805
30,31,김도영,KIA,3B,0.27200000,64,276,235,45,64,9,1,19,132,52,0,2,34,4,5,44,6,0.56200000,0.37300000,0.93500000,14,0.35600000,0.00000000,2026,0.03517715
2,3,박성한,SSG,SS,0.35500000,63,280,231,46,82,16,1,3,109,32,0,2,46,1,1,27,2,0.47200000,0.46100000,0.93300000,24,0.44200000,0.33300000,2026,0.07736969
0,1,최원준,KT,OF,0.38600000,62,297,254,54,98,19,2,5,136,37,3,3,33,0,4,40,2,0.53500000,0.45900000,0.99400000,30,0.32800000,0.00000000,2026,0.14406165
5,6,페라자,한화,OF,0.32900000,62,287,240,54,79,14,1,13,134,40,2,3,41,4,1,54,6,0.55800000,0.42500000,0.98300000,21,0.27100000,0.00000000,2026,0.07539780
3,4,레이예스,롯데,OF,0.35200000,62,277,247,28,87,15,1,9,131,44,0,3,25,5,2,32,2,0.53000000,0.41200000,0.94200000,25,0.38100000,0.00000000,2026,0.06442936


In [ ]:
# 투수 최종 후보

# 투수는 확률값 자체보다 순위 중심으로 해석
pitcher_result = pitcher_2026_model.sort_values(
    "goldenglove_probability",
    ascending=False
)

final_pitcher_candidates = pitcher_result.head(5)

final_pitcher_candidates[[
    "year", "name", "team",
    "ERA", "W", "L", "IP", "SO", "WHIP",
    "goldenglove_probability"
]]

print("투수 최종 후보: ")
final_pitcher_candidates.head(30)

투수 최종 후보: 


,rank,name,team,ERA,G,W,L,SV,HLD,WPCT,IP,H,HR,BB,HBP,SO,R,ER,WHIP,CG,SHO,QS,BSV,TBF,NP,AVG,2B,3B,SAC,SF,IBB,WP,BK,year,goldenglove_probability
2,3,류현진,한화,2.84000000,12,8,2,0,0,0.80000000,NaN,62,4,10,2,56,28,22,1.03000000,0,0,6,0,280,1059,0.23900000,14,1,7,2,1,1,0,2026,0.00181691
1,2,올러,KIA,2.66000000,13,7,5,0,0,0.58300000,NaN,52,5,25,4,86,25,24,0.95000000,1,1,8,0,320,1211,0.18200000,5,2,5,1,0,3,0,2026,0.00078523
4,5,알칸타라,키움,3.12000000,12,6,4,0,0,0.60000000,78.00000000,79,11,11,0,70,28,27,1.15000000,0,0,8,0,318,1154,0.25900000,12,2,1,1,0,2,0,2026,0.00075618
9,10,구창모,NC,3.69000000,12,6,2,0,0,0.75000000,NaN,64,8,21,2,56,31,28,1.24000000,0,0,7,0,290,1076,0.24300000,12,2,3,1,1,4,0,2026,0.00023976
3,4,최민석,두산,2.88000000,12,6,2,0,0,0.75000000,NaN,54,3,31,4,67,28,22,1.24000000,0,0,7,0,291,1098,0.21500000,13,0,3,2,0,3,1,2026,0.00020995
